# 05 · Evaluation & Visualization

**Notebook 5 of 5.** Honest metrics (**AUROC / AUPRC / balanced accuracy**) over patient-stratified
folds, a **ROC curve**, a **confusion matrix**, and a **UMAP** of the H-optimus-0 patch embeddings.

## ⚙️ Colab setup

Run the cell below **first**. It mounts Google Drive (the feature cache lives there so the expensive download+encode happens once and is reused by every notebook), and writes the course helper modules. **Use a GPU runtime** (Runtime → Change runtime type → GPU).

In [ ]:
# === Colab setup - RUN THIS CELL FIRST ===
import os, base64, gzip
# Mount Google Drive for the persistent feature cache (encode once, reuse)
try:
    from google.colab import drive; drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/pathology_mil_tcga'
except Exception:
    CACHE = os.path.abspath('./pathology_mil_tcga')   # local fallback
CACHE_BAGS = os.path.join(CACHE, 'bags'); os.makedirs(CACHE_BAGS, exist_ok=True)
MODEL_PATH = os.path.join(CACHE, 'mil_model.pt')
# Write the course helper modules so the notebook is self-contained
_FILES = {
    'mil_models.py': 'H4sIAIMjJWoC/91Z3W7byBW+11McKBdLeiXGEpBiYdQF4jS7XsDxLmInN4aXGIkjcSpqRuWQlu2sgV4Vi14uFij6PH2TPEm/M0NSpKx0nR8DRXlhUvNz5vx+55xxv9/vLVUWL00iMxutbuj9336j13Imc6mnkl59f0JiPs/lXBQmtzQzORWppB9FkZrMzG/ciqkpcyujXu88lVaSyCWGlisxLQaUS5GISSYHtDc1eS6nxR6p5SqTS6kLUSijLS2l0IWnLcU0VXo+IG2KnqAkN6uh0qCyysTU7Wl4MLmaKy0yenHy/BWvMBG9vJL5DTlpaClWlgRNxJzMrLcSxTSlmRRFmUtLdEzvf/mFXv/07vTf/0zuiApDNlOJJEilCjsgoRMK1ikUQWK1ytSUpQhxDgho22MOVjIferqiKMAahKErSAgGSysTx2mKE5kTaIfwNNSLQg9IXhe5oEPPcHAcUvU8oeMDCk4HlGDo/d9/rfdhcATVxNNMWCtt2Ov9AB5EAYEMKwnKOH/x3fNaTJbd0hQqBTOTG9apnBizoP0RBVZKYtMX07mIMiOSmFeHUa8Pn5jlZklxPCuZTByzxUxeQCcg4Y3Wq4Yg7DTt/Ii0JmFJ6+3RaFbqKW8Gm1jwba/Xe0LDL/eA2n5ER8LKTGl54NyKnsIPrmllDMbmRIE2lEmRa7ZmM7wSuVjKQuZQ6ZdlyVmKfsQ5CJQAOnhlkhJ+dOC8Aap+BSaH8GVmc8j8ON+GKQb8AU0SCyNycpTUTMk8onO3xMvJnjuBvSO2m6OayBlsp7Qq4hhmzmYDUjpO1PIA76LlP+433G88cB54QLbI8bPPiusPKFVJInW96NloXHHNjy3heEEYNeeEzRSThtWZIrbSu5pcHxL27zYUwFjkFnn/706scvMXTEBhZ/KvJceWyFh9J04ZgZenZjEc8MLX8uRN4D//DNgwZRHsR+NnYdilvNGjp19R9JS6wVUrE3G8FnlS6RKh6T36XGpr8pZSUlBsuG+F89bzxIV2xXqz+Rab0wg6CiDZ4X4YXYmsRGCrWVtVh16NBLSWvByqrdY7upUUG6oeN2q+NrIHt1GpLTQrb2WwH36IVc8tMOfFhuQTKrWCSpbUb2CvT9ZQYtYaHiTFskY9pAEwbQuVZbQ2+cJunAT4B668ImdllgXBcWRTsZIX+5cDWHEU7SN2W2OwxJWaysPjyH9sGPKQvAWt7+4eAV5GET0/4ozHWbIRfsihmLhMGHzPdpFAyiwa0Hh/9A1EQe7E9JXIFdLco+DLd3zC85qfXTAjYsVOYGYwy3WsAlr/dE4XVAidBm8pjVVI7//xL7JqvjQqCd74oUsKHw4rXbgYP/vDwKVvxOEBzZBgeNQF5INQxPnr24+EgHMWpwsBFQvbKPDmIymfVYp5EPH1TmQZfRBR0m1E6YRf6ksBz1ULZuugXgdeV0Ea0l4lHb7DLbwZhYhbjbAVmbqFP1qUY9J2yHkmKicJBOxXIcsO+AI5Wy45+YzIoOqq09Z2VAqEoXdSFzm7fNN5b6uEqrPy1ztS32cluXsJzYNFfM9zP8JFHy9RVRC5Fdu1N7UY/59LcB/McY3PMvfsr7+zf9RNj4GAg6dhBMfzSW9ACylX/HmelzJsJ6zq6BZKr6Wap+xp2P6A/Bj+Xh7cTjtRnU5H4SOln3Hkep3h2ZFLQPOtuPkaDmyB6FM5nGalRTnLUSTKa5UpgbYoM9aCCp7gpGyS1KlvFY6UWcoEXU5GLzX6Kul3j/fHo8epiVmS+OxoFyC49xmOz+RwkkOg1K1G8k0StHSa9mpBW9Cw50P1Slnf7DAeFegeF9zJOYoTUxRmyQOssgTLXPOGGitYWVkmZpiJicwynjBWFepKogTR3PviM/RlNzCzzArw5kg2fLBuMTcvM6R5BJdrH1G75CvEkONlYycURhGdohc74AnoXlk6/cHRWwy5orO0Qjubm6mEbDACG5tJ9Ddm7fOmIsWfjX0bZgphF1GjzC8Il717cbobPxGXsRXc5deD33wEpNZ7sa3+7C5omGWAq7//f2B5gzgG7d1E6Y5xW3SgRv+TINBTMExm5ge2zq43x5vN1nPhQ+9E2SK4uM/UOHR3GDH3cYjDuQw2bF628kfcHMBxUFc10MbAsxOr5LrlAPDKcx+ZdSDyHUcVg3XoDTbx2pqtozGqYYKfU9c61Z1CM7zgzlLpoONUADx6+pTG3Fed0p9QvbhWqpVpMLGgP9LooOPtFdqnkZbr+FbmxgZB2M1sIrpSch0MW7QAQE1dhe8FF1WLEPYA0rZcFpJ2lw13r4Mam3VTAZNhsdPTZetIkc+lS2mddfwJd7LBAmVdcbOSh34sM3zbVvVWad1b7Yj09uO3ei08iN7lhzrSXZ550fjMZZBe8CusM/B4wU65nYK/jabgxMaIRQTuTVCn5EoVDy10BuTw30PWz0BozRDEr08qgT67BvqcIqhbBX1CubOr799VXrjEV1/JBgU4GNCaL66KXADe9TysIMon8o0rg5ZDCxe7u4MKkegswrlOI0TYFN2obFNxIm0hkccg2DNwhMKHVm/9Dpn+weYgruvYkyalyhJ/bx5osfSXZw9Iqnt7i3XlTryNURgvhM2ac2KvEttNdS/P8ObGqH93sC1DfbtYJ7XmWH+td+hp7KItrqt7uU+mDApdwnxDJSZLlfXvUfMN4A5aTif32cP8kvnjd2wnO/ira8j/ShN+CIB/y7dpL/Pc5MGsX+qFNmtd/aPgq3d85t1XkOQ/HyV19RAZAAA=',
    'mil_utils.py': 'H4sIAIMjJWoC/71Y/47buBH+308xcNBGTmRlN2mAYlMH2AsuxQFJbnG36aEwDB1Xom2uZVEl5d0oQYB7hxboA/VN7kn6DUnJkr1JUCCp/9iVqJnhzDc/yfF4PNqqIt3VqrBJ1dDvv/2Lfl4LI/OYtjqXxVSsSm1rldFaFpU0lpbaUL2WdCHqtS70qqHXP7yiTO+MlclodGH0jcqlPRsRPaBK1EqWdWprg6elknm6WeoidxsVUmzESk6tWEp68TeyVaHqWhqK3BNdNS3/JHHSIESVqS4lPSJ5I4qdqCXxj6VtVam2onDaOMJARIXWlfUCMr2tdrVMt7I2KrPU/VjA+duffnwR05UoRJnJnESW7YzImhhfLn7CF1lnMPByLa0kDSx4d106MHKV1dNC2ZpyUQsra4ZpK2qqjM53LA3G/MpQ19lKJIUWeXolVvbXUcTmMDtBItkC2J3RB/c/VfBCiyA/F+JKFjEtpah3RlqK3vzn3/kkhlna5DamJEk+AqpLVghqkLJkJBC4fPHX8z1XJrK1V6jUtbzSekMnpw4BWGIkc5WabFPilR3PopLRGKGyNHpLabrcsaA0JbWttKlJlBAENXVpR2Gp3G0RTQKSqtE9evPj5fdnVGuTrVm6p4EKhXivioZUaWGtw9E5TpWrR85zy12ZObFkNT4DTtBAXuu+Rz5kutC81WYDaST8XtOlkZJkeaOMLrcAEd4D9/Tr/SDtVT+IM6OtnUJ1lTs8zqiL5Ivzyx++f3MZ0z4VeNm59CsrlcvlJxMvYm/GVKb8YmdPoY+U+exkwvlKBC//XUnkZ+RzTeXvYoI9/DAhfdOGKFBGzErEXCndoktdOA7Yh60TjhiWabA6QyAkRpS53iZQT+yKOsV6xJtPHNW9lo+mzz0qFHWy7lu/Lf5xcXKh4mk4Yr2AQGqx14ePboUrlYop55CQiEiXsQ6AYG2fLUHSBs2ifD7ep914EdOHsdttfEb45B+xOgYoWJovPk6OxM0PZCzmjnqRiKqSZR6poLTKWWGuHFHL6r+4XawHThgjmmjeya4WnRbOyIotZFGLyShYXji587knSJkA8K9kFDw/WXQQQRJ/xj67Uv1jJyO/dQ+jlalYGO8wV16iCiy37ICoVXbGwibzk8Wi44WXE7veLZeFjCBnjxRLuY5Z7aF/mGi/dWfN/Jr+0IZthyKYO4sNLY+t3AviIA5ow9ORl7lc7PU5Dni2GW6eLwY6fzGi+Be1ItSSDiKBeTtlgJrc7zw5iA7+NS4dj5X7BqXstS+r36AaHfTdqElrs5MxNSn649W+9oQWHPrtHZ24fSL0gpPkaUIX6ERT121i17U2GCpMSbmsuvrjNwuZZH0u+TXgbeumkpHCiBFoWaFDWl6D15l0tkTvrkPQsWliZ3QWYfd+1AJLNMXff/vnL2tVl7Khtyj7iAE3SN2qek2Ih6lA5UTryDlmN+jmSJ9Wf8hMxS5LbaZR35JOcJlWmotuWkouqtA7ahK720YTjAH8Fp3SlJpJWOv4EIaOlVP0hBDFQQLehoFrJHp7Sc7KaFyKcrwXgiEDpT5UpJVFD4+qmDZoBbMxUmElealHb7lqVHPH1isIbGxq/QQQhLmcPcWAI8uomtBDOj2Au2W+Jtb/i797FLAN0DLkSDuAjlqmd5XtBN6uVSEh9i9ub3tYejbY73qw4hk2rOOeCSNQDnPnbnnBsOLlejGU5eU9nNHpEPEeHvPrs1YERdduj4d+rwmGncfJyYCTwXAfh9h6UOW2qpvIA/oJMB11cA+YBo4ZgMlDpFqVQXytEQlqpUpMaI55dBA8kRfcMA6nCx+KiEofgA8o8g+tUfwnar+5sBxkV2WOsuuOOJz2Ggv2peYw6urKM2S7LevT9PrQ8IvPn30/NzLDd7A/YqKteKe2oML7Q3DGsKFnfUcJsjYv7yDhzXSZoXWU3D7m6Jcxf1pMnrX7HZGcgqRyNKM7UxUM6BHV+4iJnLRJgBELnGehjNHzGRfO48rnAHIVJPIcM/bPHylqyye/D+pKXR5xnBxwnAw5ll/e45Djy3scaGVlaftu8J4qOzfYyrupbL+XQ08GVPlEFnVQuyLv0ycKBX/YwSZxjxYh29G68P0kbdvhUvS1wBI5Ax46PX1+9ESH7teSOtvqchJM4WQPjY3N6TGyTFWrG1W3vLwCKt4Fp4Os94H39ZyTbzFjfPaUjsHpStQ4JVr1njv2KRBCOfj6AwnGQ3MrTB65Sw5/pLbhgD17gyP5fiZ5Wyo+y+PYXBTQa1W6YzR8wWc9utC6YFse0fl3/v+LV+ev05+/68aP2jRnn2xTTMxulRUOLvtTTXvHgpNxzQECdKw9THynejTQ3P31cSzfsVC6RJJ/b4w2Z59hZ0czKt39SguLPyweD58xPYhJVjpb29mTE+xuZo/l9E8x3eazUzl9Gh80PrYjvZVqta5nJ8mTmLJCWBtWrAMcm8kbHCln46zajdt7D7z/GbtKc6WtnL0UGJeDZ8JFgzvoHy0kZZm0lweILWHppaNxZiW1jvxeHikNmGaBD89qm5znYvuLxyCpMJ1gbJXGci2HoYWBmU7xNJeZaGa34QSLGWtglr9IqYmt26M/JGn3RZ1HH4sGH9uu7Slcdj553MHUt+BKAl6Mi7F/4lETeN4KxYZNTxN4yEPs54c15lBtGne86Q5Pstqfnrxnex034MZxEN05D4ajfSXNduevgvYXCMMjnzs4Os7hdMQ9isMNB8zh4Y8DtMOJb6BSN+7zIb290hovJodO7Q8DA4jn+xP84k4w21+hV4qdkCLS38EYCPps1djLPZRjWf+XiSsYKY6BRkP9VnzTRtNs4PyhDITWuK0FKQscuxMoq3U8Y4YN3b+H/dTDfOVY5geyhoAjBZL30uh0ZUQeYSRhmuRKZBtnOhaYwtay6oUCV/BZdyc7LB9dzfgE2CEc27MvyOdj12T7SAKB/gd63gX9EIB2Fdr06Z/1MoMviDZndJPkshbZOpokqDj8t+DKNxnR//7jwN7AUvaKzxW3U+qmiEmiarlF+fj4rM3J/RzPFwBDCxzJ4JTAtvsKOKSsDE9GyzH5UkwfZHV28jj/6N3hztKzD4zCfYfC/cVZ8mSJr1fnmDb8h/4AEr6PB6A7bTAztuX4AG2cDDZt8eshfGfl88C4q+8eOnuuwfwVIijERuhPd8dXP7ZC8/BC3LhlP981vFIsOHi+QU5Wdnj140/svrG0eXF2XNWgyP+lpvUqU/qlquRmmWHU2DbT/PZWL2s3PQapudrOTidz9Ayc3Thy+/cIHqFWwGHJ20Kbo6seh2fXIfueOR5J4v1tZ8PjaffWSmgJR/8F7LeIeLoaAAA=',
    'mil_tcga.py': 'H4sIAIMjJWoC/7Va7XLbOJb9r6fAMj9CuiVaUmI7q2l1leO4k0w5dsr2dPeUVsWCKUhim19DULI1LlfNr32A3a3aB9o36SfZcwGQBGU5ma7ppKrbFD4ugItzz70XgOM4nSSKgzJccD/fsN/+8T/sUvCYXZ+8P2YzXnKWR7mIo1SweVawcinYZ14uszhbbNinj2cszFaFFH6n83YVxTPJ5oKXq0KwvRu+kHtsXmQJK2qJd+gpejKOZoJFCV8Iydw0Y3KTQnIZhWpIb9TpMPx7/+6E/W0lig1jv/3nfzE2y+7SOOMz5su1NGVlJOVKMCkWiUhLXkZZamqG/XuW8zJcRulCibP+qQYfelleRslK9vr1rMV9WfDQkhLyEEt2z//vf2cew5JYmbFZJG87nWsub0fs/Ork7ITJ1U25yTGQUiCttHf2l+N3zI35jYhZ32OYsCm+OqmKBx7U9k7IaJGKmVLv+yxbxIKdZGjA7qJyqRReTU7PBVNLslVaootp/q6I1qLLZEatO+I+F6lESa2v70QaZlD4kudUxfb2sjQUe3uMpzOWZqW4ybJbyfrD3/7x3/0D7NZKCjWwGnDWoZ1kPCwyKaHonBe8FKzAFKJESFoC9NGTfC5GTO0sGmO2eZGFQkpMMwN4eMk4ow5qUBKuNjGSbCZiQYsposUSjealKDpRKZWyUa3noFaXpfFGdZUJj+NqU0osKStoguEqWcWYm99xAGuFvCCYr0h3QQC45VmBAVKsWOFEdkxRJrssyroslOsu+1VmaVfNtKpOVwksg0uW5p0OMBn8+PHs9IqNmbMsy1yO9vd5HvmLWeiHHHot/EW23p9HsZCOav7u+PqYfaU5wd7pfL68+PPpyXVwdvz29Aw9HpwaSs6I9bvMqTGE34NHheUXLQwK2VG9g/PjT6dUDSn9EXOUjC4bqE90f+x0Oi9Y74/7B2kDn0x2xOYRtpi2uTbYDLjrYYOAB8MsEV+kmSSL15D5gyczE3MGFQeKPgI9hAtE/irCUo5dS68tpXpdlosiCGMu5XhwANQJMRv3u9v8of8l/D642ZQCTYNhvx/09X+gL6oFBi8FsJcC+Dc8pr0GjGfAepFEaaTWHuMPy+ZaBywRJVeUO4swy1E96AOBKYjQW32kPBHmU0Z/xydYLgL5qRZmjV2mKOZRyfgchTBvMpw9ZTlClnsgVh5H8w1xljFad84lJqcYzmycR+pgSh1+tSj115hGIaBgWUpVhhmhu1S4zXIA1AEGSL9hBrZKS5RMmjWZJlHabvHgzCMRz6gq5BL0YhZU/cUiqcOaxyuBRqTAel+9x8fu7xmAdB2AdhNe2jInztVPV870dwoj2i0i5YTiQJbEkYtNW+y7BvRXpPLfPYa2oLZQMq1G0FTvOHF0onaiFu+Y7XFG1UZ17TqMQFVOBbUGaQ3Q9IaAZ5KohABq9vweWcK1hiH8z1cX5zR7kka/D2At6jewVA9OlSMuQyNCL2hJHmFc483PoUa35mLN2mO9ak3e2aocD4Z9z6ca15uozXamE4ckOVMdYNxstKmTonIoc6qccM6itLIjqYcHDaSBMbOqtRSl6z3XgYqXVEzDNZZM6kL35USDG/PpT+tKkoBKqpk4RhzN2NKq1Tia1W3tHbGakCLRJkpLd+kvMNtGv1B63/PqltFcDw7PSHOu1YJFKCE/NFQ3alEhoTRKV6IlCTODkJbKJiR+Omo6vlBBgaY9ohjTkLl8naF/LPgtgkPvy4PtGMLns5mLGTQ9q8VU9RQCzdyHCuiAHXaj+jHtGhAS8q0q9XP6jBto/lkKHjHNzU5DzijDzL4upNr6kdoUiFBUjp+t8ECv59HTSA6XmRQpdnsyrfHXgqWFQdXUR6RLimjUM52Mat839dQe8RiR+2yjltIjI0WYBstEvyqmNsKozr0Vm3HMk5sZHNiIzSytetWmt71fVswQ61FNoR2llobIpHLflR9yaw8IR1XSni+71KmI4HxfNQ73qsSEE4UtO+Sg3joTISdIaQX5uEqS/0WnBkVy2FaSK9MoeLoQrhnZa5RaFpu2ZajgvaYrsr6581BFg4/7lUt/hCVKNenxdbESDXcdIo6gmLMYPcFL4Rc8koI8FxwNUgPpek8aqfFJB66lMefuxlFS56OdKKTFhstVequW6kdEKcYRuapcYXs8YN9/j/TKGz0LZdCAaj/6Itjn/l2BMbTo9hIMIOq515XiPhTYilP1h9I0rEa0h8Hg1Y6NxxVKWA+x71NVkiJbpaR/X8ZC5O4B2wMfGUnfIVvzvkHMPPTZ9Y4U9rs6dWXuydnxp54sNzH48I+Pks2wgU6kXUXJFPculPFJnuSxGB++JpguCyGX4zddlpjPI+RMMUx2jGpYdYC0DwEVEujxUPQqszRWFa6H6mcs1kh9x9Te1dakhvRVOdCGtNJ7ypEWt8Qjxm9kq1czUzmJp9jq9vSNp4uSBcZNc58XBd8YAURwQSEWULrr9sktdvUUu6w1AkV2kpLGiSqYet7E9/0uG73ShLuUa/LH66Efrkvk71nhYryuKjm5OLu4DC7fvx1+uPpJz0UiH9bNE4FcKH0brwoXMrTQwbTWsW4e4DeXt6aLrsnimQsx9cZ02fDgQA94/eHy9OpDcHF99RcgySp5+/H8+PKvWqYlMMmKXB/pnN67VK7FfLq4/PwhODm7uDrtkt5Aq9J11Y6bjfdU+QohxhujZKILOhHqssAIp2zwxJRawi9Pry+D01+uTy/Pj8+Mnj4cfzwPjj/Dz/0SnF+cn7ZFkncLNUcRP9XFRDakd/37GDvqhh7FLDYiYcs0tk8ENrV9jplRPe1ncaW2vfJNRTQDVuHAZGUxlgSAT4Y81plZuAziflcFUOoXHNY5FFmllruN5Ocu+4Dl6rk02DPqoGHbrn7T+KY+de1ZI1dfFl1Tl/tWl5+/1kWNfI9lbjCye18RFBqxfTb08D9r2e7mS/VPyDrduLSBeQYcfc7izSJLr0H8bghJekzg7EceA2/shzHrP8XADlek1VTFey7EbGqIGgXWVKBLEFzQAcoYxZjI4WvDGnN779SxRgyvqrt4JjQ2ta0AN+Q5hRk3YLQZMydwIdKWeqZFavgIuzDLEh/A4qu4DFDuEjgaj1jPWH9M0EdFXGjpI2aKQuFac2qBjcKkPOahGGsFei3w6x4G1DdIJoIkzzWim6DqUxQWAF8P0WEvj+7B3iAvTeN9OjSIYzqsC29ptX3/gDS2SmUuwghp5czbDq8oKFEj1OhV4YZbl/uw/8+nl9d/VWdYwafPn4NfELtwyq8zHz8pZWxaY8a9e8dCwLq2HAS/6IT4W0dgt62MZ93GzJMIzlLTHDFo6a7bIYqJRH6iTPy0KLIdkVqOUNpWN7RjdF0uV8lNyqO4og/aMomv8aA/fG0FtOqkE66j6QHTomS44SY252EJNboq6RS8THguVQCIQFKfl6oNrTfiS+yiYIbJuIOuyhzpk9qTBVdzNHakZlTLgYKDZlXuz2x/n5FNfdAfpo9RRG15qkfjSrtG1eitxSu+RpL8LUKvV37rCkAZjDHUQsMR3i1jC0Rk6eJP7Kfouve+y0RyI2aU/zBojg0OXh1+i6iMkp7AzMWl7HPs3ERmrvvWrB3KiNYggLETrmbcabsToCJcqpwi0camUiAqXEe05XU7MJCkYxqNgQSjUohG/fwQHrQEL1AZcpjlvLdc3YweaE6UvOQUX/MohS9TCcxXslukfmWgDq/keCB6iFdmG4iKwgDRks4uNE81E/EF2rueX2auXqque9HavJTOmOLo7zp6dutcTy8l5MXM+xMbDl/f4z9MIl9pGi7nCa2zXr5/kkEjUrgTK7OrK6+za3XX4FoBqlV9buYg3ETwdOz2/aP+0XD4qkt2f/TmaPjv9HXUf3U4OPK+oihZzkjAcDB480YJGL7qDwZH9DU4OjqAACNh2rIrtdwuLcvwjLnPCswVkhWtaI/XdLCDFQAvyO/H0FUXzI7S8es3XweahhgGsoITviqzkEsKdlUbvypwtZjK5+pKZfyDQ4/oWY9GGZxuyQRgwYJ0FcdheW8yXn1FpvpG6VwUmLiGKraoHrsd90StuKflNtVSt0KJku5xmvXYknRMocMQ5ZajUQRyVlKmT51BlNRc2Uo6iGXvPc22CFG6dM/j7ojFPIpw13BnrgN/4DxN+aG0KFGEyf4NMZrexGozn0nW1aTQC9Ag4G53ejqI0kcVVQE2yHK2WpHTGWtguXprZInowFU9bTP27GjJfds1ZNpSM6GpGg2CNUJABmG+cuu0roEgnQYEpW0RQemH6KEEef6Sx3P0Vjd6rve8+b2AAWYFPGs+OKSwRvI1vGiOMAp2pc9GKyRqvZKtBSBsOjEJEL3Fc29UTYHCfbvRPShQt+myPd60U8T3Dfzca59dFBQL0qUEIDdiN3RVDxer7uaai17geAlVfgN3Zu6C9UWcS/dcXT0mEuuizUIlLyiUQFQ3Bm1W+Cc2OjjcRZo2V9lJ1rDf729xFoQnOY04dvbNsdZ+INfSaeKtd9WNJV3+m6MR9a2jA/oyoTx9VhfyJsJ6qS6tfaav/fQNdn14BcI4V+eSCCcRIYGMffZOXX3L5jpc3X7fcQr2gbpbIXL12ICtJF8IbNfds8F01+LgTPoJvxVYqHQtLQN4sgyyW+Wo4Q7tZkYx2406xqDVGRysGl3UiemvSNVs0XPngTZ18tIca76cPvp56dQZVNVPSZduJdEiJWMDVY0+JFnL7THrearh7MN5E5PaUXzrBNnqoA/6Ib0hG30TMW706V/gS93Sua2GwCWabaVKza1MlfPqa5eCcj+3QjAdJzbopng6zz2LPIOdhwjkNHYc0e1IDv+1Qwnru5Um2cnu92xw2PYjeUHrnDuMTeRtlE+ZjQPaGCBhBGhmLI7KEqsx73XcB0vuI6sGdnaeAtcEagcY/1JkU7k5Qw92XFlnNl39J1C63crXrKbaw8E/uA+OqtM3PU/g1r4I0tVW0TOXS/Xdj+6gfylp9SWREVXdFz4jp1ITXf+SBtUVM2kKBUZluzuqlaOR0YlTKaUqwudjt7ZbyxQaZGS3U8YemmcpE40RtZiX0+kj24UaDL01obnT+4E96JBALnlOeWENHXbfrhpMHy0w7WKXr9wkWPMnvn4G2Q9ixzA1XudRivTdYqSnVEjk0jYpNChEkq018bB/7t8L5HZCaGeBrI3OlEsRb6qTHfL3gfbuNm2337ps+UqTfT6beHZ2vodpu9+vvpBRjwaRbBfQnVwlwrPDEfVSSM+agUjpJUo1d599nAm4QPLho+qesmfeq9WErh+iETHl5glZIXrFKk0pfdcDGp/LeEqMLYpipcBQ+1nadEWuTx4T1bqrv+z11l8aHRWaEPXBSrE2xX9KOOjPTNYG/NxxHzA/N6ntZDzWB58JZRxVT/XKcJ89aTrY1fTqpCLYhhyxtNZZg/m7kxqL7G7r3Dnq6jEEImpBT170khDKe08tafIQPe7b6562SWGLEbbo4CkbuHUbyl1eTvcH4nDk9+eP7NNb25OQw94KQJ+NPp/xiM96Cnrv0DZfUlLzysD2CF9yB/+ML9hyCM94A8sVPJokmS5Zg4Sn0Vydqjcrp7m28fkfqbE3ZVAjDVPV7JGpZ6DGNLHnD7WgigIN/VFzQzxfGXrUaV9VPxdeOpUEP5RreqTn3IGgUnFH75LHzpM77Ts6Kpdr/10Ulj/TFAp3Ti/mRDwjOMnxpNmY9lbUry0spU6b7b7TN9ZLKAcyEUObAlqNVpNZuLIpUlizjIb0zijBMIpUSoXu1dtd9fCvpx4F6kdTqlCdrxFjrZBCR3SazeixWVjqJ3mXdIcNGmNhLHjBBB1FEy5rIqVXtHSHvWEuuK9+8Mv6IAnE/uWTE/omk4DevxD1b21LFezTK6ItL4eGdqxPM2Y/whjOs/JHipHV+bnbTv2d82zr5TP0YYEOedau1ag7F+VVSAHOlkzlUpTO4fA3CI233lCjT2qcBL6huEq+b9an3mfWHNhAl1a4hUIiyEIdDxkoXmrYzLcc/qw+HFOJyhdyrGLyssJtlWF12Z2gh9MyoNfR9uFpLX7SBH9Tc+sl9el7q8q6/lKnLK+G+hGQOgNB5DXPXw23BRsGMulOU7A1A+isosRZCyZUs42LS/2wXENi7nwyGNva/QbU9OLe7JgFBr/NSTRQ5/8BizDm9+0wAAA=',
}
for _n, _b in _FILES.items():
    with open(_n, 'w') as _f:
        _f.write(gzip.decompress(base64.b64decode(_b)).decode('utf-8'))
print('setup complete | feature cache:', CACHE)

In [ ]:
import numpy as np, torch
import matplotlib.pyplot as plt
from mil_tcga import load_bags, LABEL_NAME
from mil_models import build_model
from mil_utils import patient_stratified_kfold, train_one, evaluate, compute_metrics

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
data = load_bags(CACHE_BAGS)
IN_DIM = data[0]["features"].shape[1]

## 1 · Out-of-fold metrics (every slide scored by a model that didn't see it)

In [ ]:
labels = np.array([d["label"] for d in data])
cw = labels.size / (2.0 * np.maximum(np.bincount(labels, minlength=2), 1))
oof_y, oof_p, per_fold = [], [], []
for tr, va in patient_stratified_kfold(data, n_folds=4, seed=2):
    model = build_model("clam_sb", IN_DIM, 2)
    model, _ = train_one(model, data, tr, va, epochs=25, class_weights=cw, device=DEVICE, patience=8)
    m, y, p = evaluate(model, data, va, device=DEVICE, return_probs=True)
    per_fold.append(m); oof_y += y.tolist(); oof_p += p.tolist()
oof_y, oof_p = np.array(oof_y), np.array(oof_p)
print("per-fold (mean ± std):")
for k in ["auroc", "auprc", "balanced_acc"]:
    v = [m[k] for m in per_fold]; print(f"  {k:12s}: {np.nanmean(v):.3f} ± {np.nanstd(v):.3f}")
print("pooled out-of-fold:", {k: round(v, 3) for k, v in compute_metrics(oof_y, oof_p).items()})

## 2 · ROC curve and confusion matrix

In [ ]:
def roc_points(y, p):
    thr = np.unique(np.concatenate([[0], p, [1]]))[::-1]
    P, Nn = max(y.sum(), 1), max((1 - y).sum(), 1)
    tpr = [((p >= t) & (y == 1)).sum() / P for t in thr]
    fpr = [((p >= t) & (y == 0)).sum() / Nn for t in thr]
    return np.array(fpr), np.array(tpr)

fpr, tpr = roc_points(oof_y, oof_p)
auc = compute_metrics(oof_y, oof_p)["auroc"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].plot(fpr, tpr, lw=2, label=f"AUROC={auc:.3f}"); ax[0].plot([0,1],[0,1],"--",c="gray")
ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].set_title("ROC (out-of-fold)"); ax[0].legend()
pred = (oof_p >= 0.5).astype(int)
cm = np.array([[((pred==j)&(oof_y==i)).sum() for j in (0,1)] for i in (0,1)])
ax[1].imshow(cm, cmap="Purples")
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, cm[i,j], ha="center", va="center", fontsize=14,
                   color="white" if cm[i,j] > cm.max()/2 else "black")
ax[1].set_xticks([0,1], ["pred LUAD","pred LUSC"]); ax[1].set_yticks([0,1], ["LUAD","LUSC"])
ax[1].set_title("confusion @0.5")
plt.tight_layout(); plt.show()

## 3 · UMAP of H-optimus-0 patch embeddings (coloured by slide label)

In [ ]:
rng = np.random.default_rng(0)
feats, lab = [], []
for d in data:
    n = min(80, len(d["features"]))
    sel = rng.choice(len(d["features"]), n, replace=False)
    feats.append(d["features"][sel]); lab.append(np.full(n, d["label"]))
X = np.concatenate(feats); lab = np.concatenate(lab)
try:
    import umap
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(X)
    title = "UMAP of patch embeddings"
except Exception:
    Xc = X - X.mean(0); _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    emb = Xc @ Vt[:2].T; title = "PCA of patch embeddings (pip install umap-learn for UMAP)"
plt.figure(figsize=(6, 5))
for c, name in [(0, "LUAD"), (1, "LUSC")]:
    plt.scatter(emb[lab==c, 0], emb[lab==c, 1], s=8, alpha=.4, label=name)
plt.legend(); plt.title(title); plt.tight_layout(); plt.show()

### ✅ Honest-evaluation checklist
- ✅ Split by **patient**, report **mean ± std** across folds.
- ✅ Beat a **mean-pool baseline** (notebook 02).
- ✅ Report **AUROC + balanced accuracy + AUPRC** (small/imbalanced cohorts mislead on accuracy).
- ✅ Inspect **attention overlays** (notebook 04) and the **UMAP** above.
- ⚠️ ~30 slides is a *teaching* cohort — quote an **external cohort** for real claims.

That completes the TCGA pipeline: **download → encode → train → infer → post-process → evaluate.**